# Soyuz: QLoRA-дообучение DeepSeek в Google Colab

Этот ноутбук обучает LoRA-адаптер на проверенных SFT-примерах. LM Studio запускает готовый GGUF и не выполняет дообучение напрямую: после обучения адаптер нужно объединить с исходным Transformers-checkpoint и отдельно конвертировать в GGUF.

По умолчанию используется `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, который подходит для бесплатного T4 при небольшом контексте. Для более качественного результата можно указать `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`, если в Colab доступно достаточно VRAM. Seed из 15 строк предназначен только для проверки конвейера и не даёт качественного production-моделя.

In [ ]:
!pip -q install -U "transformers>=4.45,<4.48" "datasets>=2.20,<3" "peft>=0.13,<0.15" "trl==0.12.2" accelerate bitsandbytes sentencepiece


In [ ]:
import os
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("В Colab включи Runtime -> Change runtime type -> T4/L4 GPU")

MODEL_ID = os.environ.get("MODEL_ID", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
MAX_SEQ_LENGTH = int(os.environ.get("MAX_SEQ_LENGTH", "1024"))
OUTPUT_DIR = Path("/content/soyuz-deepseek-qlora")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Model:", MODEL_ID)


In [ ]:
from google.colab import files

uploaded = files.upload()
jsonl_names = [name for name in uploaded if name.endswith(".jsonl")]
if not jsonl_names:
    raise ValueError("Загрузи JSONL, созданный tools/prepare_sft_dataset.py")
DATA_PATH = Path("/content") / jsonl_names[0]
print("SFT file:", DATA_PATH)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
print(dataset)
if len(dataset) < 100:
    print("WARNING: примеров меньше 100. Это smoke-test; для качества соберите и проверьте несколько сотен примеров.")
if not dataset or "messages" not in dataset.column_names:
    raise ValueError("Каждая строка должна содержать messages из prepare_sft_dataset.py")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print("Base checkpoint loaded")


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

def format_messages(example):
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

bf16 = torch.cuda.get_device_capability(0)[0] >= 8
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    gradient_checkpointing=True,
    fp16=not bf16,
    bf16=bf16,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    formatting_func=format_messages,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Adapter saved to", OUTPUT_DIR)


In [ ]:
# Быстрый smoke-test адаптера. Это не оценка качества.
model.eval()
messages = [
    {"role": "system", "content": "Ты профессиональный переводчик. Верни только перевод."},
    {"role": "user", "content": "Переведи с en на ru: The meeting starts tomorrow."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


## Что сделать после Colab

1. Скачайте каталог `soyuz-deepseek-qlora` с адаптером и сохраните рядом `MODEL_ID`, число примеров, параметры и commit проекта.
2. Для запуска в LM Studio нужен итоговый GGUF. Адаптер из этого ноутбука сначала объединяют с исходным checkpoint, затем конвертируют инструментами `llama.cpp` в GGUF.
3. Импортируйте полученный GGUF в LM Studio, загрузите его и укажите его `id` в `AI_MODEL`.
4. Не заменяйте `seed.jsonl` автоматически сгенерированными строками: все synthetic-примеры имеют статус `unreviewed` и требуют ручной проверки.